In [ ]:
import pandas as pd 
from IPython.display import display

# 读取数据
movies = pd.read_csv('movies.csv')
display(movies.head(10))

# 创建 genre 矩阵（one-hot 编码）
genre_matrix = movies['genres'].str.get_dummies(sep='|')

# 显示矩阵
display(genre_matrix.head())

# 检查类型
print(f"genre_matrix 类型: {type(genre_matrix)}")
print(f"genre_matrix 形状: {genre_matrix.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim=cosine_similarity(genre_matrix)
print("相似度矩阵形状:",cosine_sim.shape)

In [ ]:
def get_recommendations(title):
    try:
        idx = movies[movies['title'] == title].index[0]
    except:
        return "电影库没找到这部电影，请检查拼写（包含年份）"
    
    # 获取相似度分数
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]  # 排除自己，取前5个
    
    # 提取电影索引和分数
    movie_indices = [i[0] for i in sim_scores]
    movie_scores = [i[1] for i in sim_scores]
    
    # 创建结果DataFrame
    recommendations = movies['title'].iloc[movie_indices].reset_index(drop=True)
    scores_df = pd.DataFrame(movie_scores, columns=['cosine_similarity'])
    
    # 合并显示
    result = pd.concat([recommendations, scores_df], axis=1)
    result.columns = ['推荐电影', '相似度分数']
    
    return result

In [ ]:
test_movie=movies['title'].iloc[9]
print(f"测试电影:{test_movie}")
recommendations=get_recommendations(test_movie)
print(recommendations)

In [ ]:
print(movies[movies['title']==test_movie])

In [ ]:
display(movies.head(491))

In [ ]:
#基于协同过滤的个性化推荐

In [ ]:
import pandas as pd
from IPython.display import display

In [ ]:
#读取文件
ratings=pd.read_csv('ratings.csv')
display(ratings.head(20))

In [ ]:
#将表格转换成矩阵
user_movie_matrix=ratings.pivot(index='userId',columns='movieId',values='rating')

In [ ]:
#矩阵的列是电影（有至少10个评分）
user_movie_matrix=user_movie_matrix.dropna(thresh=10,axis=1)

In [ ]:
#矩阵的行是用户（有看过至少20部）
user_movie_matrix=user_movie_matrix.dropna(thresh=20,axis=0)

In [ ]:
user_movie_matrix_filled=user_movie_matrix.fillna(0)
print("过滤后的矩阵形状",user_movie_matrix_filled.shape)
print("前五行数据",user_movie_matrix_filled.head(5))

In [ ]:
#计算用户相似度
from sklearn.metrics.pairwise import cosine_similarity
user_sim=cosine_similarity(user_movie_matrix_filled)
#转换成Dataframe，给定标签便于索引
user_sim_df=pd.DataFrame(user_sim,
                         index=user_movie_matrix.index,
                         columns=user_movie_matrix.index)
print("用户相似度矩阵（前五名用户）:")
print(user_sim_df.head(5))

In [ ]:
def get_user_recommendations(user_id, top_n=5):
    # 1. 找到和该用户最像的前 10 个人 (用方括号，变量名对齐)
    similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:11].index
    
    # 2. 看看这些“邻居”都评价过哪些电影 
    neighbor_ratings = ratings[ratings['userId'].isin(similar_users)]
    
    # 3. 排除掉该用户已经看过的电影
    user_watched = ratings[ratings['userId'] == user_id]['movieId']
    recommendations = neighbor_ratings[~neighbor_ratings['movieId'].isin(user_watched)]
    
    # 4. 计算这些电影在邻居中的平均分，并排序
    res = recommendations.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(top_n)
    
    # 5. 联动 movies 表展示结果
    return movies[movies['movieId'].isin(res.index)][['title', 'genres']]

In [ ]:
print(f"为用户{user_movie_matrix.index[0]}推荐的电影列表:")
get_user_recommendations(user_movie_matrix.index[0])

In [ ]:
#去中心化
#计算每个用户的平均分
user_mean=user_movie_matrix.mean(axis=1)
#去中心化
user_movie_matrix_centered=user_movie_matrix.sub(user_mean,axis=0)
#以0代替nan
user_movie_matrix_final=user_movie_matrix_centered.fillna(0)
#打印矩阵并查看
print(user_movie_matrix_final.head())

In [ ]:
#计算Pearson相关系数
from sklearn.metrics.pairwise import cosine_similarity
#计算更新后的相似度
user_sim_improved=cosine_similarity(user_movie_matrix_final)
#转换成DataFrame
user_sim_df_improved=pd.DataFrame(user_sim_improved,
                               index=user_movie_matrix.index,
                               columns=user_movie_matrix.index)
print(user_sim_df_improved.head())

In [ ]:
#编写混合推荐
def get_hybird_recommendations(user_id,top_n=5):
    #1.用户看过超过20部
    if user_id in user_sim_df_improved.index:
        #1.找最像的10名用户
        similar_users=user_sim_df_improved[user_id].sort_values(ascending=False)[1:11].index
        #2.找10名用户看过但该用户没看过的电影
        user_watched=ratings[ratings['userId']==user_id]['movieId']
        neighbor_ratings=ratings[ratings['userId'].isin(similar_users)]
        recommendations=neighbor_ratings[~neighbor_ratings['movieId'].isin(user_watched)]
        #3.按平均排序
        res=recommendations.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(top_n)
        print(f">>>为用户{user_id}匹配喜好口味:")
        return movies[movies['movieId'].isin(res.index)][['title','genres']]
    #2.用户不在矩阵中
    else:
        #1.按热度排序推荐
        popular_movies=ratings.groupby('movieId')['rating'].count().sort_values(ascending=False).head(top_n)
        print(f">>>识别为新用户{user_id},正在进行热度补偿推荐:")
        return movies[movies['movieId'].isin(popular_movies.index)][['title','genres']]

In [ ]:
#测试1：1号活跃用户
print("为活跃用户生成的个性化菜单")
res_active=get_hybird_recommendations(1)
display(res_active)

In [ ]:
#测试2：模拟一个新用户9999（不在矩阵中）
print("为新用户生成的推荐菜单")
res_new=get_hybird_recommendations(9999)
display(res_new)

In [ ]:
!pip install openai

In [ ]:
import openai
from openai import OpenAI

# 检查是否安装成功，如果不报错就说明 OK
print("OpenAI 库加载成功！")

# 初始化客户端
# 这里的 api_key 记得换成你申请到的那个字符串
# 如果是 DeepSeek，base_url 设为  https://api.deepseek.com 
client = OpenAI(
    api_key="your api-key", 
    base_url="https://api.deepseek.com"
)

In [ ]:
def get_ai_reason(user_history, target_movie):
    """
    user_history: 用户过去喜欢的电影（字符串）
    target_movie: 算法通过皮尔逊系数算出来的推荐电影
    """
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
           # model="deepseek-v4-pro",
            messages=[
                {"role": "system", "content": "你是一个幽默且专业的电影评论家。"},
                {"role": "user", "content": f"用户以前喜欢看：{user_history}。现在算法为他推荐了《{target_movie}》。请写一段20字左右的推荐理由，要突出电影之间的关联。"}
            ],
            stream=False
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"生成理由失败：{e}"

In [ ]:
# 模拟你算法算出来的结果
my_recommendations = ["星际穿越", "地心引力", "火星救援"]
history = "《盗梦空间》, 《敦刻尔克》"

print(f"--- 基于皮尔逊算法为您生成的推荐清单 --- \n")

for movie in my_recommendations:
    reason = get_ai_reason(history, movie)
    print(f"🎬 电影：{movie}")
    print(f"💬 推荐理由：{reason}")
    print("-" * 30)
